# Build and inspect an energy network

Optionally build the energy networks, then inspect **one** of them: summary, map, node and edge tables, validation report and PyPSA network.

The build turns the inputs checked in `../00-data-review/00_data_review.ipynb` into three networks, each a PyPSA file with GeoParquet map layers:

- `base-mauritius`: CEB substations moved onto the CEB 66 kV routes, with the power plants attached.
- `inferred-osm-<region>`: OpenStreetMap substations and power plants, connected by the roads near lit areas (the estimated distribution network).
- `inferred-provided-<region>`: CEB substations, power plants and 66 kV lines, connected by the same roads.

The region is `energy.region` in `config/energy/energy.yaml` (`mauritius-rodrigues`). Voltages and capacities of the inferred lines are placeholders: `v_nom_kv` is empty in the map layers and `model_v_nom_kv` holds the value the model uses.

## How the build runs

This notebook builds nothing unless you set `RUN_BUILD = True`. The target `build_energy_networks` in `workflow/0-preprocess/energy.smk` asks for the three `<network>.nc` files, and Snakemake runs these rules:

1. `fetch_energy_osm` downloads the OpenStreetMap roads, power features and island outlines to `data/incoming/Infrastructure/Energy/OpenStreetMap/<region>/`, if a file is missing.
2. `prepare_energy_assets` cleans the CEB shapefiles and demand workbook into `data/processed/energy/provided/`, placing some plants by their OpenStreetMap name.
3. `build_base_energy_network` writes `base-mauritius`.
4. `fetch_energy_nightlights` (if an image is missing), `build_energy_nightlight_composite` and `build_energy_nightlight_targets` turn the monthly VIIRS images into lit-pixel points (`targets.geoparquet`).
5. `build_inferred_energy_network` runs twice, writing `inferred-osm-<region>` and `inferred-provided-<region>`. Both read the roads and targets; the osm network also reads the OpenStreetMap power features, the provided network the cleaned CEB tables.

Each build rule runs `workflow/0-preprocess/energy_build_network.py`, which calls `energy.build.build_network(source, ...)` with settings from `config/energy/energy.yaml`. A network is written to `data/processed/energy/networks/<name>/` (`<name>.nc`, `<name>_metadata.json`, and `geoparquet/` with the node and edge layers that `h.load_layers` reads) and to `data/out/energy/<name>/` (`generators.csv`, `lines.csv`, `validation.json`).

**From the terminal** (repo root, `mu-star` environment):

```shell
snakemake -c1 build_energy_networks      # all three networks
snakemake -n build_energy_networks       # dry run: what would run and why
snakemake -c1 -F build_energy_networks   # rerun every rule, downloads included
# one network: ask for the file it writes
snakemake -c1 data/processed/energy/networks/base-mauritius/base-mauritius.nc
snakemake -c1 data/processed/energy/networks/inferred-osm-mauritius-rodrigues/inferred-osm-mauritius-rodrigues.nc
snakemake -c1 data/processed/energy/networks/inferred-provided-mauritius-rodrigues/inferred-provided-mauritius-rodrigues.nc
```

Only `build_energy_networks` can be run by name; the other rules are asked for by the path of a file they write.

**From this notebook**: set `RUN_BUILD = True` in the next cell. It runs the same target from the repo root with this kernel's Python. Rerun the cells below afterwards.

In [ ]:
# Set RUN_BUILD = True to run Snakemake from the repo root with this kernel's Python.
import pathlib
import subprocess
import sys

RUN_BUILD = False
# All three networks. For one network, use the file it writes, e.g.
# "data/processed/energy/networks/base-mauritius/base-mauritius.nc".
TARGET = "build_energy_networks"
EXTRA_ARGS: list[str] = []  # e.g. ["-n"] for a dry run, ["-F"] to rerun every rule

if RUN_BUILD:
    repo_root = next(p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents) if (p / "pyproject.toml").exists())
    subprocess.run([sys.executable, "-m", "snakemake", "-c1", TARGET, *EXTRA_ARGS], cwd=repo_root, check=True)

## Pick a network

The next cell lists the networks that are built (those with `geoparquet/` layers and a manifest) and picks one by index. Change `PRODUCT_INDEX` and rerun from there to see another.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers finds the workflow's files and draws them.
import _helpers as h  # noqa: E402

products = h.available_products()
if not products:
    raise RuntimeError(
        f"No built products under {h.NETWORKS_DIR}. Run `{h.BUILD_ALL}` (or set RUN_BUILD = True above) and re-run."
    )
print("Built products:")
display(h.list_products())

# Change the index to inspect another network.
PRODUCT_INDEX = 0
PRODUCT = products[PRODUCT_INDEX]
print("\nSelected PRODUCT =", PRODUCT)

## Summary

Counts, total line length, and which electrical values are placeholders (from the manifest).

In [ ]:
summary = h.summarise(PRODUCT)
for key in ("product", "methodology", "nodes", "edges", "line_length_km", "crs", "electrical_values_note"):
    print(f"{key:24s} {summary[key]}")
print("\nnodes by kind:  ", summary["node_kinds"])
print("edges by source:", summary["edge_sources"])

## Interactive map

Hover over a bus to see its attributes (Plotly on an OpenStreetMap basemap). Every edge is drawn, coloured by `source`; in the inferred networks the roads are the estimated distribution lines, drawn under the 66 kV lines and connections. Only substations and generators are marked. Use `roads=False` to hide the roads, or `clip=h.MAURITIUS_BBOX` to show one island (Rodrigues is about 560 km from Mauritius, so the full view is mostly sea). `h.plot_network(PRODUCT)` draws a static matplotlib version.

In [ ]:
h.explore_network(PRODUCT)

## Node and edge tables

These GeoParquet layers are what the viewer shows. Nodes have `bus_id`, `kind` (`substation`, `generator`, `transformer`, `distribution_node`, or `junction` in base-mauritius), `source` and `model_v_nom_kv`; a power asset connected to a road also has `anchor_status` and `anchor_distance_m` (its distance from that road). Edges have `source` (for example `osm` for roads, `provided_transmission`, and `substation_anchor` or `generator_anchor` for connections), `bus0`, `bus1`, `length_km` and `model_v_nom_kv`. `is_inferred` marks what the estimate added. In the inferred networks `v_nom_kv` and `s_nom_mva` are empty; the placeholder values are in `model_v_nom_kv` and `model_s_nom_mva`.

In [ ]:
nodes, edges = h.load_layers(PRODUCT)
node_columns = [
    "bus_id",
    "name",
    "kind",
    "source",
    "model_v_nom_kv",
    "anchor_status",
    "anchor_distance_m",
    "is_inferred",
]
edge_columns = ["line_id", "source", "bus0", "bus1", "length_km", "model_v_nom_kv", "model_s_nom_mva", "is_inferred"]
print("Power terminals (nodes that are not inferred road vertices):")
display(h.anchor_nodes(nodes)[node_columns].head(10))
print("Edges:")
display(edges[edge_columns].head(10))
print("Edges by source:")
display(edges.groupby("source", dropna=False)["length_km"].agg(count="size", length_km="sum").round(1))

## Validation report

`data/out/energy/<network>/validation.json` has totals, an `errors` list (empty for a usable network) and `warnings`, for example when the line length or generation capacity differs from CEB's published totals by more than the set tolerance. The same warnings are logged during the build. `status` is `valid`, `valid_with_warnings` or `invalid`.

In [ ]:
if h.found(h.OUT_DIR / PRODUCT / "validation.json", h.BUILD_ALL):
    validation = h.load_validation(PRODUCT)
    print("status:", validation["status"])
    print("totals:", validation["totals"])
    for error in validation.get("errors", []):
        print("  error:  ", error)
    for warning in validation.get("warnings", []):
        print("  warning:", warning)
    for check, result in validation.get("checks", {}).items():
        print(f"  check {check}: {result.get('status')}")

## PyPSA view

The same network as a PyPSA file (`<network>.nc`): buses, lines, generators, and transformers where inferred-provided joins the CEB 66 kV lines to the 11 kV roads. Capacities are fixed; nothing is extendable.

In [ ]:
n = h.load_pypsa(PRODUCT)
print(n)
print(f"{len(n.buses)} buses, {len(n.lines)} lines, {len(n.transformers)} transformers, {len(n.generators)} generators")
display(n.buses.head())
display(n.lines.head())